In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install transformers datasets peft accelerate -q

In [3]:
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments

from peft import LoraConfig, TaskType, get_peft_model

In [4]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

What is the encoded numeric label for the row at index 150?


In [5]:
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train["label"] = train["answer"].map(label_map)

print("Answer 1")
print(train.iloc[150]["label"])

Answer 1
2


Q2. Prompt-Option Formatting
For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?


In [6]:
text = str(train.iloc[0]["prompt"]) + " [SEP] " + str(train.iloc[0]["B"])

print(text)

print("Answer 2")
print(len(text))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Answer 2
407


Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?


In [7]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

choices = []


for option in ["A", "B", "C", "D", "E"]:
    choices.append(str(train.iloc[0]["prompt"]) + " [SEP] " + str(train.iloc[0][option]))

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].reshape(1, 5, 128)

print(input_ids.shape)

print("Answer 3")
print(input_ids.shape[1])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

torch.Size([1, 5, 128])
Answer 3
5


Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?


In [8]:
all_input_ids = []

for i in range(16):

    choices = []

    for option in ["A","B","C","D","E"]:
        choices.append(str(train.iloc[i]["prompt"]) + " [SEP] " + str(train.iloc[i][option]))

    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    all_input_ids.append(encoding["input_ids"])

all_input_ids = torch.stack(all_input_ids)

print(all_input_ids.shape)

print("Answer 4")
print(all_input_ids.numel())

torch.Size([16, 5, 128])
Answer 4
10240


Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?


In [9]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

choices = []

for option in ["A","B","C","D","E"]:
    choices.append(str(train.iloc[0]["prompt"]) + " [SEP] " + str(train.iloc[0][option]))

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

outputs = model(
    input_ids=encoding["input_ids"].unsqueeze(0),
    attention_mask=encoding["attention_mask"].unsqueeze(0)
)

print(outputs.logits.shape)

print("Answer 5")
print(outputs.logits.shape[1])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 5])
Answer 5
5


Q6. Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?


In [10]:
label = torch.tensor([train.iloc[0]["label"]])

outputs = model(
    input_ids=encoding["input_ids"].unsqueeze(0),
    attention_mask=encoding["attention_mask"].unsqueeze(0),
    labels=label
)

print(outputs.loss)

print("Answer 6")
print(outputs.loss.dim())

tensor(1.6129, grad_fn=<NllLossBackward0>)
Answer 6
0


In [11]:
# Had problems with package version, so uninstalled the library
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


Q7. LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?


In [12]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, config)

count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Answer 7")
print(count)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Answer 7
295681


Q8. Hugging Face Dataset Preparation
Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?


In [13]:
rows = []

for i in range(100):

    choices = []

    for option in ["A","B","C","D","E"]:
        choices.append(str(train.iloc[i]["prompt"]) + " [SEP] " + str(train.iloc[i][option]))

    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    rows.append({
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": train.iloc[i]["label"]
    })

dataset = Dataset.from_list(rows)

print("Answer 8")
print(len(dataset[0]["input_ids"]))

Answer 8
5


Q9. Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?


In [22]:
small_rows = []

for i in range(32):

    choices = []

    for option in ["A","B","C","D","E"]:
        choices.append(str(train.iloc[i]["prompt"]) + " [SEP] " +str(train.iloc[i][option]))

    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=64
    )

    small_rows.append({
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": train.iloc[i]["label"]
    })

small_dataset = Dataset.from_list(small_rows)
small_dataset.set_format(type="torch")

model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, config)

args = TrainingArguments(
    output_dir="output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=small_dataset
)

result = trainer.train()

print("Answer 9")
print(result.global_step)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss


Answer 9
4


Q10. Probability Assigned to Option E After Fine-Tuning
Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.


In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

choices = []

for option in ["A","B","C","D","E"]:
    choices.append(str(train.iloc[0]["prompt"]) + " [SEP] " + str(train.iloc[0][option]))

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)


input_ids = encoding["input_ids"].unsqueeze(0).to(device)
attention_mask = encoding["attention_mask"].unsqueeze(0).to(device)

model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

probabilities = torch.softmax(outputs.logits, dim=1)


print("Answeer 10")
print(round(probabilities[0][4].item(), 4))

Answeer 10
0.2093
